In [1]:
import os
import os.path as osp
import sys 
sys.path.append("..")
import torch 
from torch import nn
from torch.utils.data import DataLoader
from cacti.utils.mask import generate_masks
from cacti.utils.utils import save_single_image,get_device_info,load_checkpoints
from cacti.utils.metrics import compare_psnr,compare_ssim
from cacti.utils.config import Config
from cacti.models.builder import build_model
from cacti.datasets.builder import build_dataset 
from cacti.utils.logger import Logger
from torch.cuda.amp import autocast
import numpy as np 
import argparse 
import time
import einops 
import cv2
import numpy as np 
import scipy.io as scio 
from torch.utils.data import Dataset 
import os 
import os.path as osp
import h5py

In [2]:
class Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        in_ch = cfg.in_ch
        color_ch = cfg.color_ch
        color_ch_G = color_ch[0] #gray
        color_ch_C = color_ch[1] #color
        cfg0 = {k: (v[0] if k == 'color_ch' else v) for k, v in cfg.items()}
        cfg1 = {k: (v[1] if k == 'color_ch' else v) for k, v in cfg.items()}

        self.model0 = build_model(cfg0)
        self.model1 = build_model(cfg1)

        #(1,1,8,128,128)-->(1,3,8,128,128)
        self.up_conv_G2C = nn.Sequential(
            nn.Linear(color_ch_G, 3),
            nn.LeakyReLU(),
            nn.Linear(3, 3),
            nn.LeakyReLU(),
            nn.Linear(3, color_ch_C),
            nn.LeakyReLU(),
        )

        #(1,6,8,128,128)-->(1,3,8,128,128)
        self.output_fuse = nn.Sequential(
            nn.Conv3d(2*color_ch_C, 3, kernel_size=(3,7,7), stride=1,padding=(1,3,3)),
            nn.LeakyReLU(),
            nn.Conv3d(3, 3, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(),
            nn.Conv3d(3, color_ch_C, kernel_size=3, stride=(1,1,1), padding=1),
            nn.LeakyReLU(),
        )
        #(1,256,8,128,128)-->(1,256,8,128,128)
        self.fem0 = nn.Sequential(
            nn.Conv3d(in_ch*4, in_ch*4, kernel_size=3, stride=1, padding=1),
            nn.ELU(),
            nn.Conv3d(in_ch*4, in_ch*4, kernel_size=1, stride=1),
            nn.ELU(),
            nn.Conv3d(in_ch*4, in_ch*4, kernel_size=3, stride=1, padding=1),
            nn.ELU(),
        )
        self.fem1 = nn.Sequential(
            nn.Conv3d(in_ch*4, in_ch*4, kernel_size=3, stride=1, padding=1),
            nn.ELU(),
            nn.Conv3d(in_ch*4, in_ch*4, kernel_size=1, stride=1),
            nn.ELU(),
            nn.Conv3d(in_ch*4, in_ch*4, kernel_size=3, stride=1, padding=1),
            nn.ELU(),
        )

        #(1,256*3,8,128,128)-->(1,1,8,128,128)
        self.up_conv = nn.Conv3d(in_ch*4*2,in_ch*8*2,1,1)
        self.up = nn.PixelShuffle(2)
        self.vrm = nn.Sequential(
            nn.Conv3d(in_ch*4, in_ch*2, kernel_size=3, stride=1, padding=1),
            nn.ELU(),
            nn.Conv3d(in_ch*2, in_ch, kernel_size=1, stride=1),
            nn.ELU(),
            nn.Conv3d(in_ch, color_ch_C, kernel_size=3, stride=1, padding=1),
        )

    def forward(self,y_list,Phi_list,Phi_s_list):
        out_list = []
        y0,y1 = y_list
        Phi0,Phi1 = Phi_list
        Phi0_s,Phi1_s = Phi_s_list

        output00,output01 = self.model0(y0,Phi0,Phi0_s)
        output10,output11 = self.model1(y1,Phi1,Phi1_s)

        output00 = self.fem0(output00)
        output10 = self.fem1(output10)

        output0 = torch.cat((output00,output10),dim=1)       
        output0 = self.up_conv(output0)
        output0 = einops.rearrange(output0,"b c t h w-> b t c h w")
        output0 = self.up(output0)
        output0 = einops.rearrange(output0,"b t c h w-> b c t h w")
        output0 = self.vrm(output0)

        # if self.color_ch == 3:
        #     output1 = torch.cat((output01,output11,output21),dim=0)
        #     output1 = einops.rearrange(output1,"b c t h w-> c b t h w")
        #     output1 = self.output_fuse(output1)
        #     output1 = einops.rearrange(output1,"c b t h w-> b c t h w")
        # else:
        # (1,1,8,128,128) --> (1,8,128,128,1) --> (1,8,128,128,3) --> (1,3,8,128,128)
        output01 = einops.rearrange(output01,"b c t h w-> b t h w c")
        output01 = self.up_conv_G2C(output01)
        output01 = einops.rearrange(output01,"b t h w c-> b c t h w")

        output1 = torch.cat((output01,output11),dim=1)
        output1 = self.output_fuse(output1)

        out = output0 + output1

        # if self.color_ch!=3:
        #     out = out.squeeze(1)
        out_list.append(out)
        return output11, out_list

In [3]:
class MiddleScaleData(Dataset):
    def __init__(self,data_root,**kwargs):
        self.data_dir= data_root
        self.data_list = os.listdir(data_root)
        
        self.mask = kwargs["mask"]
        self.ratio,self.mask_h,self.mask_w= self.mask[0].shape

        r = np.array([[1, 0], [0, 0]])
        g1 = np.array([[0, 1], [0, 0]])
        g2 = np.array([[0, 0], [1, 0]])
        b = np.array([[0, 0], [0, 1]])
        self.rgb2raw = np.zeros([3, self.mask_h, self.mask_w])
        self.rgb2raw[0, :, :] = np.tile(r, (self.mask_h // 2, self.mask_w // 2))
        self.rgb2raw[1, :, :] = np.tile(g1, (self.mask_h // 2, self.mask_w // 2)) + np.tile(g2, (
            self.mask_h // 2, self.mask_w // 2))
        self.rgb2raw[2, :, :] = np.tile(b, (self.mask_h // 2, self.mask_w // 2))

    def __getitem__(self,index):
        with h5py.File(osp.join(self.data_dir,self.data_list[index]), 'r') as file:
            imgs = np.array(file['orig'])

        imgs = imgs.astype(np.float32)/255
        imgs = imgs[...,0:self.mask_h,0:self.mask_w] #(32, 3, 512, 512)
        
        # imgs_bayer = np.sum(imgs*self.rgb2raw[np.newaxis,:,:,:],axis=1) #(32, 512, 512)
        pic_gt = np.zeros([imgs.shape[0] // self.ratio, self.ratio, 3, self.mask_h, self.mask_w]) #([4, 8, 3, 512, 512])
        for ii in range(len(imgs)):
            if ii % self.ratio == 0:
                meas0_t = np.zeros([self.mask_h, self.mask_w])
                meas1_t = np.zeros([self.mask_h, self.mask_w])
                n = 0
            img_t = imgs[ii, :, :, :] # [3,512,512]
            # img_bayer_t = imgs_bayer[ii, :, :] # [512,512]
            mask0_t = self.mask[0][n, :, :] # [512,512]
            mask1_t = self.mask[1][n, :, :]

            pic_gt[ii // self.ratio, n, :, :, :] = img_t # [4, 8, 3, 512, 512]
            n += 1

            Y0 = cv2.cvtColor(img_t.transpose(1,2,0),cv2.COLOR_RGB2YCrCb)[:,:,0] # [512,512]
            meas0_t = meas0_t + np.multiply(mask0_t, Y0) # [512,512]
            Y1 = np.sum(img_t*self.rgb2raw,axis=0) # [512,512]
            meas1_t = meas1_t + np.multiply(mask1_t, Y1) # [512,512]

            if ii == (self.ratio-1):
                meas0_t = np.expand_dims(meas0_t, 0)
                meas0 = meas0_t
                meas1_t = np.expand_dims(meas1_t, 0)
                meas1 = meas1_t
            elif (ii + 1) % self.ratio == 0 and ii != (self.ratio-1):
                meas0_t = np.expand_dims(meas0_t, 0)
                meas0 = np.concatenate((meas0, meas0_t), axis=0) # [4, 512, 512]
                meas1_t = np.expand_dims(meas1_t, 0)
                meas1 = np.concatenate((meas1, meas1_t), axis=0) # [4, 512, 512]

        pic_gt = pic_gt.transpose(0,2,1,3,4)
        return meas0,meas1,pic_gt
    def __len__(self,):
        return len(self.data_list)

In [4]:
cfg = Config.fromfile('./configs/EfficientSCI/efficientsci_base.py')
cfg.train_data.type = 'DavisMixData'
cfg.train_data.gene_meas.type = ['GenerationGrayMeas', 'GenerationBayerMeas']
cfg.gene_meas.type = ['GenerationGrayMeas', 'GenerationBayerMeas']
cfg.model.color_ch = [1, 3]

cfg.weights0 = 'checkpoints/ckpt0_color.pth'
cfg.weights1 = 'checkpoints/ckpt1_color.pth'

In [5]:
work_dir = './simulate_eval'
device = 'cuda:0'

# data_root = '../../datasets/test_datasets/color_smallscale'
data_root = '../../datasets/test_datasets/color_middlescale'
data_name_list = os.listdir(data_root)
with h5py.File(osp.join(data_root,data_name_list[0]), 'r') as file:
    mask0 = np.array(file['mask_bayer']).astype(np.float32)
mask0.shape

(8, 512, 512)

In [6]:
work_dir = './simulate_eval'
device = 'cuda:0'

data_root = '../../datasets/test_datasets/color_middlescale'
data_name_list = os.listdir(data_root)
with h5py.File(osp.join(data_root,data_name_list[0]), 'r') as file:
    mask0 = np.array(file['mask_bayer']).astype(np.float32)
mask1 = 1 - mask0

mask0_s = np.sum(mask0, axis=0)
mask0_s[mask0_s==0] = 1
mask1_s = np.sum(mask1, axis=0)
mask1_s[mask1_s==0] = 1

cr = mask0.shape[0]
# scio.savemat('mask_middlescale.mat',{'mask':mask0.transpose(1,2,0)})
# mask0 = scio.loadmat('mask_middlescale.mat')['mask'].transpose(2,1,0)

########################################################################
test_data = MiddleScaleData(data_root,**{"mask":(mask0,mask1)})
data_loader = DataLoader(test_data,batch_size=1,shuffle=False)
########################################################################

In [7]:
cfg_model0 = {k: (3 if k == 'color_ch' else v) for k, v in cfg.model.items()}
model0 = build_model(cfg_model0).to(device)
resume_dict0 = torch.load(cfg.weights0)
if "model_state_dict" not in resume_dict0.keys():
    model_state_dict0 = resume_dict0
else:
    model_state_dict0 = resume_dict0["model_state_dict"]
load_checkpoints(model0,model_state_dict0,strict=True)

/tmp/ipykernel_66022/1765724544.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  resume_dict0 = torch.load(cfg.weights0)


In [8]:
cfg_model1 = cfg.model
model1 = Model(cfg_model1).to(device)
resume_dict1 = torch.load(cfg.weights1)
if "model_state_dict" not in resume_dict1.keys():
    model_state_dict1 = resume_dict1
else:
    model_state_dict1 = resume_dict1["model_state_dict"]
load_checkpoints(model1,model_state_dict1,strict=True)

/tmp/ipykernel_66022/1011326839.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  resume_dict1 = torch.load(cfg.weights1)


In [9]:
Phi0 = einops.repeat(mask0,'cr h w->b cr h w',b=1)
Phi0_s = einops.repeat(mask0_s,'h w->b 1 h w',b=1)

Phi0 = torch.from_numpy(Phi0).to(device)
Phi0_s = torch.from_numpy(Phi0_s).to(device)

Phi1 = einops.repeat(mask1,'cr h w->b cr h w',b=1)
Phi1_s = einops.repeat(mask1_s,'h w->b 1 h w',b=1)

Phi1 = torch.from_numpy(Phi1).to(device)
Phi1_s = torch.from_numpy(Phi1_s).to(device)

In [10]:
output_dict0 ={}
output_dict1 ={}
output_dict2 ={}
output_dict3 ={}

meas_dict0 = {}
meas_dict1 = {}
meas_dict2 = {}
meas_dict3 = {}

gt_dict = {}

psnr_dict0,ssim_dict0 = {},{}
psnr_dict1,ssim_dict1 = {},{}
psnr_dict2,ssim_dict2 = {},{}
psnr_dict3,ssim_dict3 = {},{}

psnr_list0,ssim_list0 = [],[]
psnr_list1,ssim_list1 = [],[]
psnr_list2,ssim_list2 = [],[]
psnr_list3,ssim_list3 = [],[]
# sum_time=0.0
# time_count = 0
for data_iter,data in enumerate(data_loader):
    meas0, meas1, gt = data
    gt = gt[0].numpy() # (4,3,8,512,512)
    
    meas0 = meas0[0].float().to(device)
    meas1 = meas1[0].float().to(device)
    batch_size = meas0.shape[0]
    name = test_data.data_list[data_iter]
    if "_" in name:
        _name,_ = name.split("_")
    else:
        _name,_ = name.split(".")
    print(gt.shape, name)
    meas_dict0[_name] = meas0.cpu().numpy().astype(np.float32)
    meas_dict1[_name] = meas1.cpu().numpy().astype(np.float32)
    if data_root == '../../datasets/test_datasets/color_middlescale':
        gt = gt.transpose(0,1,2,4,3)
        data_dim = 512
    elif data_root == '../../datasets/test_datasets/color_smallscale':
        data_dim =256
    gt_dict[_name] = gt

    gt_list = []
    batch_output0 = []
    batch_output1 = []
    batch_output2 = []
    batch_output3 = []
    
    psnr0,ssim0 = 0,0
    psnr1,ssim1 = 0,0
    psnr2,ssim2 = 0,0
    psnr3,ssim3 = 0,0

    dict_pnp = scio.loadmat(f'../GAP-TV+PnP-FFDNet+PnP-FastDVDnet/results/savedmat_color{data_dim}/gapfastdvdnet_demosaic_{_name}_bayer8_sigma6_all7.mat')
    
    pics_pnp_ffdnet = (dict_pnp['vgap_ffd_color_demosaic_bayer']*255).astype(np.uint8).transpose(2,0,1)#(32, 512, 512)
    pics_pnp_ffdnet = np.array([cv2.cvtColor(pic_pnp_ffdnet, cv2.COLOR_BAYER_BG2RGB) 
                                 for pic_pnp_ffdnet in pics_pnp_ffdnet], dtype=np.uint8)
    pics_pnp_ffdnet = pics_pnp_ffdnet.reshape(pics_pnp_ffdnet.shape[0]//8, 8, *pics_pnp_ffdnet.shape[-3:])
    pics_pnp_ffdnet = pics_pnp_ffdnet.transpose(0,4,1,2,3) #(4,3,8,512,512)

    pics_pnp_fastdvd = (dict_pnp['vgap_fastdvd_color_demosaic_bayer']*255).astype(np.uint8).transpose(2,0,1)
    pics_pnp_fastdvd = np.array([cv2.cvtColor(pic_pnp_fastdvd, cv2.COLOR_BAYER_BG2RGB) 
                                 for pic_pnp_fastdvd in pics_pnp_fastdvd], dtype=np.uint8)
    pics_pnp_fastdvd = pics_pnp_fastdvd.reshape(pics_pnp_fastdvd.shape[0]//8, 8, *pics_pnp_fastdvd.shape[-3:])
    pics_pnp_fastdvd = pics_pnp_fastdvd.transpose(0,4,1,2,3) #(4,3,8,512,512)

    for ii in range(batch_size):
        single_gt = gt[ii]
        single_meas0 = meas0[ii].unsqueeze(0).unsqueeze(0)
        single_meas1 = meas1[ii].unsqueeze(0).unsqueeze(0)

        with torch.no_grad():
            torch.cuda.synchronize()
            # start = time.time()

            # outputs = model(single_meas, Phi, Phi_s)
            # single_meas0 = single_meas0.permute(0,1,3,2)
            # single_meas1 = single_meas1.permute(0,1,3,2)
            # Phi0 = Phi0.permute(0,1,3,2)
            # Phi1 = Phi1.permute(0,1,3,2)
            # Phi0_s = Phi0_s.permute(0,1,3,2)
            # Phi1_s = Phi1_s.permute(0,1,3,2)
            outputs0 = model0(single_meas1, Phi1, Phi1_s)
            _, outputs1 = model1((single_meas0,single_meas1), (Phi0,Phi1), (Phi0_s,Phi1_s))
            
            torch.cuda.synchronize()
            # end = time.time()
            # run_time = end - start
            # if ii>0:
            #     sum_time += run_time
            #     time_count += 1
        if not isinstance(outputs0,list):
            outputs0 = [outputs0]
        output0 = outputs0[-1].squeeze(1)
        output0 = output0[0].cpu().numpy().astype(np.float32) #(3,8,512,512)
        
        if not isinstance(outputs1,list):
            outputs1 = [outputs1]
        output1 = outputs1[-1][0].cpu().numpy().astype(np.float32) #(3,8,512,512)
        if data_root == '../../datasets/test_datasets/color_middlescale':
            output0 = output0.transpose(0,1,3,2)
            output1 = output1.transpose(0,1,3,2)
        
        output2 = pics_pnp_ffdnet[ii]
        output3 = pics_pnp_fastdvd[ii]

        batch_output0.append(output0)
        batch_output1.append(output1)
        batch_output2.append(output2)
        batch_output3.append(output3)
        
        for jj in range(cr):
            # if output0.shape[0]==3:
            #     per_frame_out0 = output0[:,jj]
            #     rgb2raw = test_data.rgb2raw
            #     per_frame_out0 = np.sum(per_frame_out0*rgb2raw,axis=0)
            # else:
            per_frame_out0 = output0[...,jj,:,:]
            per_frame_out1 = output1[...,jj,:,:]
            per_frame_out2 = output2[...,jj,:,:]
            per_frame_out3 = output3[...,jj,:,:]
            per_frame_gt = single_gt[...,jj,:,:]
            psnr0 += compare_psnr(per_frame_gt*255,per_frame_out0*255)
            ssim0 += compare_ssim(per_frame_gt*255,per_frame_out0*255)
            psnr1 += compare_psnr(per_frame_gt*255,per_frame_out1*255)
            ssim1 += compare_ssim(per_frame_gt*255,per_frame_out1*255)
            psnr2 += compare_psnr(per_frame_gt*255,per_frame_out2)
            ssim2 += compare_ssim(per_frame_gt*255,per_frame_out2)
            psnr3 += compare_psnr(per_frame_gt*255,per_frame_out3)
            ssim3 += compare_ssim(per_frame_gt*255,per_frame_out3)

    meas_num = len(batch_output0)
    psnr0 = psnr0 / (meas_num* cr)
    ssim0 = ssim0 / (meas_num* cr)
    psnr1 = psnr1 / (meas_num* cr)
    ssim1 = ssim1 / (meas_num* cr)
    psnr2 = psnr2 / (meas_num* cr)
    ssim2 = ssim2 / (meas_num* cr)
    psnr3 = psnr3 / (meas_num* cr)
    ssim3 = ssim3 / (meas_num* cr)

    psnr_list0.append(psnr0)
    ssim_list0.append(ssim0)

    psnr_list1.append(psnr1)
    ssim_list1.append(ssim1)

    psnr_list2.append(psnr2)
    ssim_list2.append(ssim2)

    psnr_list3.append(psnr3)
    ssim_list3.append(ssim3)


    psnr_dict0[_name] = psnr0
    ssim_dict0[_name] = ssim0

    psnr_dict1[_name] = psnr1
    ssim_dict1[_name] = ssim1

    psnr_dict2[_name] = psnr2
    ssim_dict2[_name] = ssim2

    psnr_dict3[_name] = psnr3
    ssim_dict3[_name] = ssim3

    output_dict0[_name] = np.array(batch_output0)
    output_dict1[_name] = np.array(batch_output1)
    output_dict2[_name] = np.array(batch_output2)
    output_dict3[_name] = np.array(batch_output3)

(4, 3, 8, 512, 512) Beauty_bayer.mat
(4, 3, 8, 512, 512) Bosphorus_bayer.mat
(4, 3, 8, 512, 512) Jockey_bayer.mat
(4, 3, 8, 512, 512) Runner_bayer.mat
(4, 3, 8, 512, 512) ShakeNDry_bayer.mat
(4, 3, 8, 512, 512) Traffic_bayer.mat


In [11]:
psnr_dict0, np.array([v for v in psnr_dict0.values()]).mean()

({'Beauty': 33.74631290202693,
  'Bosphorus': 34.09894755110055,
  'Jockey': 35.61837286404753,
  'Runner': 37.32289579451472,
  'ShakeNDry': 33.20847274042126,
  'Traffic': 28.613935504432543},
 33.768156226090589)

In [12]:
psnr_dict1, np.array([v for v in psnr_dict1.values()]).mean()

({'Beauty': 37.407289190258645,
  'Bosphorus': 41.017090182612414,
  'Jockey': 38.84528158317299,
  'Runner': 42.97849661508808,
  'ShakeNDry': 36.45413923858876,
  'Traffic': 31.571067670957774},
 38.045560746779778)

In [13]:
psnr_dict2, np.array([v for v in psnr_dict2.values()]).mean()

({'Beauty': 34.34145474582331,
  'Bosphorus': 33.11194879695698,
  'Jockey': 34.74682227357777,
  'Runner': 34.3195630510497,
  'ShakeNDry': 32.551158658339645,
  'Traffic': 22.129481843651043},
 31.866738228233071)

In [14]:
psnr_dict3, np.array([v for v in psnr_dict3.values()]).mean()

({'Beauty': 35.092282406724664,
  'Bosphorus': 36.57635808075411,
  'Jockey': 35.22617697220383,
  'Runner': 35.13659323353078,
  'ShakeNDry': 33.672002047978594,
  'Traffic': 21.99589857100812},
 32.949885218700018)